# Loading and inspecting a `System`

The one settled idea in `py-gromos`'s design (see `PLAN.md` Priority 3): a GROMOS
simulation is two files — a **topology** (`.top`, connectivity/parameters) and a
**configuration** (`.cnf`, positions/velocities/box) — mirrored as two Rust objects,
paired by a `System`:

```python
system = System.from_files("water.topo", "water.cnf")
```

`System(topo, conf)` validates that atom counts match at construction — the earliest
possible catch of a `.top`/`.cnf` mismatch, which used to silently corrupt downstream
runs.

This notebook loads real reference systems (the same fixtures
`cargo test -p gromos-md --test test_gromosXX_references` validates against, in
`crates/gromos-md/tests/gromosXX_references/`) and inspects them: atom counts,
masses, charges, and one real structural-analysis plot (O–O radial distribution
function of bulk water).

In [ ]:
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt

from gromos import System, Topology, Configuration, rdf

REF_DIR = "../../crates/gromos-md/tests/gromosXX_references"


## 1. A simple system: bulk SPC water

`water_216_box` is 648 atoms (216 SPC water molecules) in a rectangular box, no
bonded (angle/dihedral) terms — the simplest non-trivial reference system.

In [ ]:
water = System.from_files(
    f"{REF_DIR}/water_216_box/water_216_box.topo",
    f"{REF_DIR}/water_216_box/water_216_box.conf",
)
print(water)
print(f"n_atoms = {water.n_atoms}")
print(f"charge  = {water.charge}")
print(f"box     = {water.box} nm")


## 2. A real molecule: alanine dipeptide in a water shell

`aladip_solvated` pairs a 12-atom solute (alanine dipeptide, all bonded terms:
bond/angle/dihedral/improper) with 60 SPC solvent atoms — 72 atoms total.
`system.topology.n_solute_atoms` / `n_solvent_atoms` come from the Dim 10
solute/solvent role split (`PLAN.md` §2.0).

In [ ]:
aladip = System.from_files(
    f"{REF_DIR}/shared/aladip.topo",
    f"{REF_DIR}/aladip_vacuum/aladip_vacuum.conf",
)
print(aladip)
print(f"n_atoms         = {aladip.n_atoms}")
print(f"n_solute_atoms  = {aladip.topology.n_solute_atoms}")
print(f"n_solvent_atoms = {aladip.topology.n_solvent_atoms}")
print(f"charge          = {aladip.charge}")


`aladip_vacuum` has no solvent (`n_solvent_atoms == 0`) — it's the same 12-atom
solute topology as `aladip_solvated`, just without the water shell. `System.from_files`
only works here because the topology and configuration already agree on atom count
(12 solute atoms, no solvent block to replicate). The *solvated* variant needs a
different loading path — see the last section of this notebook.

## 3. Plot — mass histogram

Real per-atom data from the topology: `system.topology.masses` (amu). Water is two
masses (O, H); alanine dipeptide's solute has a wider spread (C, N, O, H).

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=water.topology.masses, name="water_216_box", nbinsx=20))
fig.add_trace(go.Histogram(x=aladip.topology.masses, name="aladip_vacuum", nbinsx=20))
fig.update_layout(
    barmode="overlay",
    xaxis_title="mass (amu)",
    yaxis_title="count",
    title="Per-atom mass distribution",
)
fig.update_traces(opacity=0.7)
fig


## 4. Plot — O–O radial distribution function of bulk water

A real physical observable computed from the loaded positions: the free `gromos.rdf()`
function on the O atoms of `water_216_box` (SPC ordering is O, H, H per molecule, so
O indices are every 3rd atom). The first peak of SPC water's O–O RDF is expected
around 0.28-0.32 nm.

In [ ]:
positions = water.positions.astype(np.float32)
o_indices = list(range(0, water.n_atoms, 3))

r, g_r = rdf(positions, o_indices, o_indices, n_bins=100, r_max=1.0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=r, y=g_r, mode="lines", name="g_OO(r)"))
fig.update_layout(
    xaxis_title="r (nm)",
    yaxis_title="g(r)",
    title="O-O radial distribution function, water_216_box",
)
fig


In [ ]:
peak_r = r[np.argmax(g_r)]
print(f"first peak at r = {peak_r:.3f} nm (SPC water: ~0.28 nm)")
assert 0.25 < peak_r < 0.32, f"unexpected RDF peak at {peak_r} nm"


## 5. Round-trip: write and reload

`System.write()` writes the configuration side back out to a `.cnf` file. Reloading
it should reproduce the same positions.

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmp:
    out_path = os.path.join(tmp, "water_roundtrip.cnf")
    water.write(out_path)
    reloaded = System.from_files(
        f"{REF_DIR}/water_216_box/water_216_box.topo",
        out_path,
    )
    assert np.allclose(reloaded.positions, water.positions)
    print("round-trip OK:", reloaded.n_atoms, "atoms, positions match")


## 6. The atom-count guard

Constructing a `System` from a mismatched topology/configuration pair raises
`ValueError` immediately, rather than silently producing garbage downstream — this is
exactly the bug class described in `PLAN.md` Priority 3 (`.top`/`.cnf` mismatches that
used to corrupt `vsomm_modeler` runs).

In [ ]:
mismatched_topo = Topology(f"{REF_DIR}/water_216_box/water_216_box.topo")
mismatched_conf = Configuration(f"{REF_DIR}/nacl_pair/nacl_pair.conf")

try:
    System(mismatched_topo, mismatched_conf)
    raise AssertionError("expected a ValueError")
except ValueError as e:
    print(f"ValueError raised as expected: {e}")


## 7. Composing a `System` *without* `from_files`

`System.from_files()` is a convenience for the common case, but it eagerly checks
`topo.n_atoms == conf.n_atoms` — which breaks down for a topology that hasn't been
**solvated** yet. `aladip_solvated`'s topology (`shared/aladip.topo`) only describes
the 12-atom solute; the matching configuration (`shared/aladip.conf`) has 72 atoms
(12 solute + 60 solvent, i.e. `NSM=20` SPC molecules replicated at load time in
GROMOS). `System.from_files()` on that pair raises immediately — the topology and
configuration genuinely disagree until the topology is solvated.

The manual path — `Topology`, `.solvate(nsm)`, `Configuration`, then the `System(topo,
conf)` constructor directly — handles this:

In [ ]:
# This fails: topology (12 atoms) vs configuration (72 atoms) mismatch.
try:
    System.from_files(f"{REF_DIR}/shared/aladip.topo", f"{REF_DIR}/shared/aladip.conf")
    raise AssertionError("expected a ValueError")
except ValueError as e:
    print(f"from_files() raised as expected: {e}")


In [ ]:
# The manual path: load Topology and Configuration separately, solvate, then compose.
topo = Topology(f"{REF_DIR}/shared/aladip.topo")
print(f"before solvate: {topo.n_atoms} atoms")

topo.solvate(20)  # NSM=20 from aladip_solvated.in: 12 solute + 20*3 solvent = 72
print(f"after solvate:  {topo.n_atoms} atoms "
      f"({topo.n_solute_atoms} solute + {topo.n_solvent_atoms} solvent)")

conf = Configuration(f"{REF_DIR}/shared/aladip.conf")
aladip_solvated = System(topo, conf)  # constructor, not from_files()
print(aladip_solvated)


This is also the path to build a `Simulation` directly from `Topology` +
`Configuration` + `Recipe` objects, bypassing `System` entirely — used in
`02_short_md.ipynb`.

## Summary

- `System.from_files()` for the common case: matching topology/configuration files.
- `System(topo, conf)` for everything else — including topologies that need
  `.solvate(nsm)` before the atom counts line up.
- Both paths validate atom counts and raise `ValueError` on mismatch.

Next: [`02_short_md.ipynb`](02_short_md.ipynb) — running MD with native Python
parameters and analyzing the results with `EnergyTimeseries`.